# ЛизаАлерт GPS-треки — EDA

**FTP:** `maps.lizaalert.ru:21` → `maps/tracks-archive` (14 000+ поисков, с 2011) + `maps/tracks` (703 активных)
**Формат:** OziExplorer `.plt`
**Цель:** понять структуру данных, статистику, пригодность для SAR-моделей

In [ ]:
import ftplib, io, re, os
from pathlib import Path
from datetime import datetime, timedelta
from math import radians, sin, cos, sqrt, atan2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print('folium не установлен: pip install folium')

FTP_HOST = 'maps.lizaalert.ru'
FTP_PORT = 21
FTP_USER = 'lizaalert'
FTP_PASS = '123'

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('OK, все импорты загружены')

## 1. Масштаб архива

In [ ]:
def ftp_connect():
    ftp = ftplib.FTP()
    ftp.connect(FTP_HOST, FTP_PORT, timeout=30)
    ftp.login(FTP_USER, FTP_PASS)
    ftp.set_pasv(True)
    return ftp

def list_dir(ftp, path):
    items = []
    lines = []
    try:
        ftp.retrlines(f'LIST {path}', lines.append)
    except Exception as e:
        print(f'LIST {path}: {e}')
        return items
    for line in lines:
        parts = line.split(None, 8)
        if len(parts) < 9:
            continue
        name = parts[8].strip()
        if name in ('.', '..'):
            continue
        is_dir = parts[0].startswith('d')
        size = 0 if is_dir else int(parts[4])
        items.append((name, is_dir, size))
    return items

ftp = ftp_connect()
for root in ['maps/tracks-archive', 'maps/tracks']:
    items = list_dir(ftp, root)
    dirs  = [i for i in items if i[1]]
    print(f'{root}:')
    print(f'  папок: {len(dirs)}')
    print(f'  от {dirs[0][0]}')
    print(f'  до {dirs[-1][0]}')
ftp.quit()

## 2. Скачиваем выборку

In [ ]:
# Параметры выборки - берём последние N миссий из archive
N_MISSIONS = 50      # сколько поисков взять
ROOT = 'maps/tracks-archive'

def download_file(ftp, remote, local: Path):
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.exists():
        return
    buf = io.BytesIO()
    ftp.retrbinary(f'RETR {remote}', buf.write)
    local.write_bytes(buf.getvalue())

ftp = ftp_connect()
missions = sorted(
    [i for i in list_dir(ftp, ROOT) if i[1] and re.match(r'\d{4}-\d{2}-\d{2}', i[0])],
    key=lambda x: x[0],
    reverse=True
)[:N_MISSIONS]

print(f'Берём {len(missions)} поисков: от {missions[-1][0]} до {missions[0][0]}')

meta = []   # список всех скачанных файлов
for mname, _, _ in missions:
    tracks_in_mission = [
        i for i in list_dir(ftp, f'{ROOT}/{mname}')
        if not i[1] and i[0].lower().endswith('.plt')
    ]
    for fname, _, fsize in tracks_in_mission:
        local = DATA_DIR / mname / fname
        try:
            download_file(ftp, f'{ROOT}/{mname}/{fname}', local)
            meta.append({'mission': mname, 'tracker': Path(fname).stem,
                         'file': fname, 'path': local, 'size_kb': fsize/1024})
        except Exception as e:
            print(f'  ошибка {mname}/{fname}: {e}')

ftp.quit()
print(f'\nСкачано: {len(meta)} .plt файлов')

## 3. Парсер PLT

In [ ]:
OLE_EPOCH = datetime(1899, 12, 30)

def parse_plt(path: Path) -> pd.DataFrame:
    text = path.read_text(encoding='latin-1', errors='ignore')
    lines = text.splitlines()

    data_start = 6
    for i in range(3, min(10, len(lines))):
        if re.match(r'^\s*\d+\s*$', lines[i]):
            data_start = i + 1
            break

    records = []
    for line in lines[data_start:]:
        parts = line.split(',')
        if len(parts) < 5:
            continue
        try:
            lat = float(parts[0])
            lon = float(parts[1])
            if abs(lat) < 0.01 or abs(lon) < 0.01:
                continue
            new_seg = int(parts[2])
            alt_ft  = float(parts[3])
            alt_m   = None if alt_ft < -700 else round(alt_ft * 0.3048, 1)
            ole = float(parts[4])
            ts = (OLE_EPOCH + timedelta(days=ole)) if ole > 30000 else None
            records.append({'lat': lat, 'lon': lon, 'new_seg': new_seg,
                            'alt_m': alt_m, 'ts': ts})
        except:
            continue
    return pd.DataFrame(records)

test = parse_plt(Path(meta[0]['path']))
print(f'Файл: {meta[0]["file"]}')
print(f'Точек: {len(test)}')
test.head()

## 4. Загружаем все в один DataFrame

In [ ]:
all_dfs = []
for rec in meta:
    try:
        df = parse_plt(rec['path'])
        if len(df) < 3:
            continue
        df['mission'] = rec['mission']
        df['tracker'] = rec['tracker']
        all_dfs.append(df)
    except Exception as e:
        print(f'  parse error {rec["file"]}: {e}')

tracks = pd.concat(all_dfs, ignore_index=True)
print(f'Всего точек:    {len(tracks):,}')
print(f'Поисков:        {tracks["mission"].nunique()}')
print(f'Трекеров:       {tracks["tracker"].nunique()}')
print(f'С timestamp:    {tracks["ts"].notna().sum():,} ({100*tracks["ts"].notna().mean():.1f}%)')
print(f'С высотой:      {tracks["alt_m"].notna().sum():,} ({100*tracks["alt_m"].notna().mean():.1f}%)')
tracks.describe()

## 5. Статистика по трекам

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

def track_stats(df):
    df = df.sort_values('ts') if df['ts'].notna().any() else df.reset_index(drop=True)
    pts = df[['lat','lon']].values
    dist_m = sum(haversine_m(*pts[i], *pts[i+1]) for i in range(len(pts)-1))
    dur_min = None
    speed_kmh = None
    if df['ts'].notna().all() and len(df) >= 2:
        dur_s = (df['ts'].max() - df['ts'].min()).total_seconds()
        dur_min = dur_s / 60
        if dur_s > 0:
            speed_kmh = (dist_m / 1000) / (dur_s / 3600)
    return pd.Series({
        'n_pts': len(df),
        'dist_km': round(dist_m / 1000, 3),
        'dur_min': round(dur_min, 1) if dur_min else None,
        'speed_kmh': round(speed_kmh, 2) if speed_kmh else None,
        'lat_start': float(pts[0, 0]),
        'lon_start': float(pts[0, 1]),
    })

stats = (
    tracks.groupby(['mission','tracker'])
    .apply(track_stats, include_groups=False)
    .reset_index()
)

print(stats[['dist_km','dur_min','speed_kmh','n_pts']].describe().round(2))
stats.head(10)

In [ ]:
# Статистика по поискам
per_mission = (
    stats.groupby('mission').agg(
        n_trackers=('tracker','nunique'),
        total_km=('dist_km','sum'),
        mean_km=('dist_km','mean'),
        median_speed=('speed_kmh','median'),
    ).reset_index()
    .sort_values('n_trackers', ascending=False)
)
print(f'Поисков: {len(per_mission)}')
print(per_mission.head(15).to_string(index=False))

## 6. Визуализация

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f'ЛизаАлерт GPS-треки - EDA ({tracks["mission"].nunique()} поисков)', fontsize=13, fontweight='bold')

# 1. Длина треков
ax = axes[0,0]
stats['dist_km'].clip(upper=50).hist(bins=50, ax=ax, color='#3b82f6', edgecolor='none')
ax.axvline(stats['dist_km'].median(), color='red', lw=1.5, ls='--',
           label=f'медиана {stats["dist_km"].median():.1f} км')
ax.set_title('Длина трека (км)'); ax.set_xlabel('км'); ax.legend(fontsize=8)

# 2. Продолжительность
ax = axes[0,1]
dur = stats['dur_min'].dropna()
dur.clip(upper=600).hist(bins=50, ax=ax, color='#22c55e', edgecolor='none')
if len(dur):
    ax.axvline(dur.median(), color='red', lw=1.5, ls='--',
               label=f'медиана {dur.median():.0f} мин')
    ax.legend(fontsize=8)
ax.set_title('Длительность (мин)'); ax.set_xlabel('мин')

# 3. Скорость
ax = axes[0,2]
spd = stats['speed_kmh'].dropna()
spd_clean = spd[(spd > 0.1) & (spd < 15)]
if len(spd_clean):
    spd_clean.hist(bins=50, ax=ax, color='#a855f7', edgecolor='none')
    ax.axvline(spd_clean.median(), color='red', lw=1.5, ls='--',
               label=f'медиана {spd_clean.median():.1f} км/ч')
    ax.legend(fontsize=8)
ax.set_title('Средняя скорость (км/ч)'); ax.set_xlabel('км/ч')

# 4. Треков на поиск
ax = axes[1,0]
per_mission['n_trackers'].hist(bins=30, ax=ax, color='#f97316', edgecolor='none')
ax.set_title('Треков на поиск'); ax.set_xlabel('кол-во трекеров')

# 5. Точки на карте
ax = axes[1,1]
sample = tracks.sample(min(80_000, len(tracks)), random_state=42)
ax.scatter(sample.lon, sample.lat, s=0.2, alpha=0.1, c='#ef4444', rasterized=True)
ax.set_title(f'GPS-точки (sample {min(80000,len(tracks)):,})')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
ax.set_aspect('equal')

# 6. Общая покрытая дистанция по поискам (топ-20)
ax = axes[1,2]
top = per_mission.head(20)
ax.barh(range(len(top)), top['total_km'], color='#eab308')
ax.set_yticks(range(len(top)))
ax.set_yticklabels([m[-15:] for m in top['mission']], fontsize=7)
ax.set_title('Топ-20: суммарный км/поиск'); ax.set_xlabel('км (все трекеры)')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('figures/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: figures/eda_overview.png')

In [ ]:
# Треки одного поиска (самого большого)
mission_name = per_mission.iloc[0]['mission']
mdf = tracks[tracks['mission'] == mission_name].copy()
trackers = mdf['tracker'].unique()
colors = cm.tab20(np.linspace(0, 1, max(len(trackers), 1)))

fig, ax = plt.subplots(figsize=(11, 9))
for tracker, color in zip(trackers, colors):
    tdf = mdf[mdf['tracker'] == tracker]
    tdf = tdf.sort_values('ts') if tdf['ts'].notna().any() else tdf.reset_index(drop=True)
    # Разбиваем по new_seg
    segs, curr = [], []
    for _, row in tdf.iterrows():
        if row['new_seg'] == 1 and curr:
            segs.append(curr); curr = []
        curr.append((row['lon'], row['lat']))
    if curr: segs.append(curr)
    for seg in segs:
        xs, ys = zip(*seg)
        ax.plot(xs, ys, color=color, linewidth=1.5, alpha=0.85)
    if len(tdf):
        ax.scatter(tdf.iloc[0]['lon'], tdf.iloc[0]['lat'], s=60, color=color,
                   zorder=5, marker='o', label=tracker[:20])

ax.set_title(f'Поиск: {mission_name}  ({len(trackers)} трекеров)', fontsize=11)
ax.set_xlabel('Долгота'); ax.set_ylabel('Широта')
ax.set_aspect('equal')
if len(trackers) <= 15:
    ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f'figures/mission_{mission_name}.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Интерактивная карта (Folium)

In [ ]:
if HAS_FOLIUM:
    mission_name = per_mission.iloc[0]['mission']
    mdf = tracks[tracks['mission'] == mission_name]
    center = [mdf['lat'].mean(), mdf['lon'].mean()]
    m = folium.Map(location=center, zoom_start=13)

    pal = ['red','blue','green','purple','orange','darkred','cadetblue',
           'pink','lightblue','beige','lightgreen','gray','darkblue','darkgreen']
    for i, tracker in enumerate(mdf['tracker'].unique()):
        tdf = mdf[mdf['tracker'] == tracker]
        tdf = tdf.sort_values('ts') if tdf['ts'].notna().any() else tdf
        coords = tdf[['lat','lon']].values.tolist()
        if len(coords) >= 2:
            folium.PolyLine(coords, color=pal[i % len(pal)],
                            weight=2.5, opacity=0.8, tooltip=tracker).add_to(m)
        if coords:
            folium.CircleMarker(
                coords[0], radius=6, color='black', weight=1,
                fill=True, fill_color=pal[i % len(pal)], fill_opacity=0.9,
                tooltip=f'* {tracker}'
            ).add_to(m)
    out = f'map_{mission_name}.html'
    m.save(out)
    print(f'Сохранено: {out}  (открой в браузере)')
    display(m)
else:
    print('pip install folium')

In [ ]:
spd_clean = stats['speed_kmh'].dropna()
spd_clean = spd_clean[(spd_clean > 0.1) & (spd_clean < 15)]

print(f'Датасет: {tracks["mission"].nunique()} поисков, {len(stats)} треков, {len(tracks):,} GPS-точек')
if len(spd_clean):
    print(f'Скорость: медиана {spd_clean.median():.2f} км/ч  |  10-90 пц: {spd_clean.quantile(0.1):.2f}-{spd_clean.quantile(0.9):.2f} км/ч')